# MCP Integration - External Service Tools

## Purpose
Learn how to integrate external services into your agents using the Model Context Protocol (MCP). MCP enables agents to access databases, APIs, documentation systems, and other external resources through standardized connectors.

## Key Concepts
- **MCP (Model Context Protocol)**: Standard for connecting agents to external services
- **HostedMCPTool**: Wrapper for AWS Bedrock MCP connectors
- **Connector ID**: AWS ARN identifying the MCP gateway
- **allowed_tools**: Optional list to restrict which MCP tools are available

## Installation

In [ ]:
#!pip install openai
#!pip install openai-agents
#!pip install aws-bedrock-token-generator

## Authentication Setup

In [ ]:
model_id = "openai.gpt-5.5"

In [ ]:
from openai import AsyncOpenAI
from agents import (
    set_default_openai_client,
    set_default_openai_api,
    set_tracing_disabled,
)
from aws_bedrock_token_generator import provide_token

client = AsyncOpenAI(
    api_key=provide_token(),
    base_url="https://bedrock-mantle.us-east-1.api.aws/openai/v1",
    project="default"
)

set_default_openai_client(client)
set_default_openai_api("responses")
set_tracing_disabled(True)  # OpenAI-platform tracing can't reach Mantle

## Import Libraries

Import `HostedMCPTool` for MCP integration:

In [ ]:
import asyncio

from agents import Agent, Runner, function_tool, HostedMCPTool
from openai.types.responses import ResponseTextDeltaEvent

## Step 0: Configure the AgentCore Gateway with the MCP Endpoint

This section stands up the gateway that fronts the **AWS Knowledge MCP Server** at
`https://aws-mcp.us-east-1.api.aws/mcp`. That endpoint is a fully managed, AWS-hosted MCP server that
exposes AWS's own knowledge as MCP tools — up-to-date AWS documentation search/read, AWS API &
CloudFormation **regional availability**, the full **region inventory**, and documentation
recommendations.

The steps below:

1. **Step 0a** — create the gateway IAM **service role** dynamically (idempotent).
2. **`create_gateway`** — create the MCP gateway (`protocolType="MCP"`) with **IAM inbound auth**
   (`authorizerType="AWS_IAM"`), then **wait until the gateway is `READY`**.
3. **`create_gateway_target`** — attach the MCP endpoint as an **MCP server** target with outbound auth
   via the **gateway IAM role** (`credentialProviderType="GATEWAY_IAM_ROLE"`).

> ⚠️ **Ordering matters**: a target cannot be added while the gateway is still `CREATING`. Always wait
> for the gateway to reach `READY` before calling `create_gateway_target`.

The values mirror the console **Edit target** form:

| Console field | Value | API mapping |
|---|---|---|
| Target protocol | MCP target | `targetConfiguration.mcp` |
| Target name | `aws-mcp-target` | `name` |
| Passthrough | Do not use passthrough (default aggregated) | default behavior |
| Target type | MCP server | `mcp.mcpServer` |
| MCP endpoint | `https://aws-mcp.us-east-1.api.aws/mcp` | `mcpServer.endpoint` |
| Outbound Auth | IAM role | `credentialProviderType="GATEWAY_IAM_ROLE"` |
| **Inbound Auth** (gateway) | IAM | `authorizerType="AWS_IAM"` |

The resulting gateway ARN is what you pass as `connector_id` in Step 1.

### Step 0a: Create the Gateway service role (dynamically)

The gateway needs an IAM **service role** that AgentCore assumes on its behalf. Rather than assume it
pre-exists, we create it here with the required **trust policy** (principal
`bedrock-agentcore.amazonaws.com`, action `sts:AssumeRole`). This step is **idempotent** — if the role
already exists it is reused.

Because our outbound authorization is **No authorization (`NONE`)**, the role needs no extra resource
permissions to reach the MCP endpoint. We attach the AWS-managed `BedrockAgentCoreFullAccess` policy for
gateway operations — tighten this for production.

> **Note**: The trust policy's `Condition` (SourceAccount/SourceArn) is omitted at creation because the
> gateway ARN isn't known yet. As a best practice, add it back after the gateway is created.

In [ ]:
import json
import boto3
from botocore.exceptions import ClientError

REGION = "us-east-1"
GATEWAY_ROLE_NAME = "AgentCoreGatewayRole"

iam = boto3.client("iam")
account_id = boto3.client("sts", region_name=REGION).get_caller_identity()["Account"]

# Trust policy: allow the AgentCore Gateway service to assume this role.
# Condition omitted at creation (gateway ARN not yet known) — add SourceAccount/SourceArn later.
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "GatewayAssumeRolePolicy",
            "Effect": "Allow",
            "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
}

try:
    create_role_response = iam.create_role(
        RoleName=GATEWAY_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="AgentCore Gateway service role for the AWS Knowledge MCP gateway",
    )
    GATEWAY_ROLE_ARN = create_role_response["Role"]["Arn"]
    print("Created role:", GATEWAY_ROLE_ARN)
except ClientError as e:
    if e.response["Error"]["Code"] == "EntityAlreadyExists":
        GATEWAY_ROLE_ARN = iam.get_role(RoleName=GATEWAY_ROLE_NAME)["Role"]["Arn"]
        print("Role already exists, reusing:", GATEWAY_ROLE_ARN)
    else:
        raise

# Attach a policy for gateway operations (managed policy for simplicity; scope down for prod).
iam.attach_role_policy(
    RoleName=GATEWAY_ROLE_NAME,
    PolicyArn="arn:aws:iam::aws:policy/BedrockAgentCoreFullAccess",
)
print("Attached BedrockAgentCoreFullAccess")

# IAM is eventually consistent — give the new role a moment to propagate before use.
import time
time.sleep(10)
print("GATEWAY_ROLE_ARN ->", GATEWAY_ROLE_ARN)

In [ ]:
import time
import random
random_number = random.randrange(10,99)

MCP_ENDPOINT = "https://aws-mcp.us-east-1.api.aws/mcp"

# GATEWAY_ROLE_ARN comes from Step 0a (created dynamically above).

gateway_client = boto3.client("bedrock-agentcore-control", region_name=REGION)

READY = "READY"
FAILED_STATES = {"FAILED", "DELETING", "DELETE_FAILED"}

def _wait_until_ready(describe_fn, label, timeout=300, interval=10):
    """Poll describe_fn() until status == READY (or raise on failure / timeout)."""
    deadline = time.time() + timeout
    while True:
        resp = describe_fn()
        status = resp["status"]
        print(f"{label} status: {status}")
        if status == READY:
            return resp
        if status in FAILED_STATES:
            raise RuntimeError(f"{label} failed: {status} -> {resp.get('statusReasons', resp)}")
        if time.time() > deadline:
            raise TimeoutError(f"{label} not READY after {timeout}s (last status: {status})")
        time.sleep(interval)

# 1) Create the MCP gateway with IAM-based INBOUND authorization.
#    authorizerType="AWS_IAM" -> callers authenticate to the gateway via AWS SigV4 (IAM).
create_gateway_response = gateway_client.create_gateway(
    name="aws-mcp-" + str(random_number),
    roleArn=GATEWAY_ROLE_ARN,
    protocolType="MCP",
    authorizerType="AWS_IAM",
    description="Gateway fronting the AWS Knowledge MCP endpoint (IAM inbound)",
)
gateway_id = create_gateway_response["gatewayId"]
gateway_arn = create_gateway_response["gatewayArn"]
print("Gateway ID :", gateway_id)
print("Gateway ARN:", gateway_arn)

# IMPORTANT: the gateway must be READY before a target can be added.
# Adding a target while the gateway is still CREATING fails, so wait here first.
_wait_until_ready(
    lambda: gateway_client.get_gateway(gatewayIdentifier=gateway_id),
    label="Gateway",
)

# 2) Attach the MCP endpoint as a gateway target (gateway is now READY).
#    Mirrors the console 'Edit target' form:
#      - Target protocol : MCP target
#      - Target type     : MCP server
#      - Passthrough     : Do not use passthrough (default aggregated)
#      - Outbound Auth   : IAM role -> credentialProviderType="GATEWAY_IAM_ROLE"
create_target_response = gateway_client.create_gateway_target(
    gatewayIdentifier=gateway_id,
    name="aws-mcp-target",
    description="AWS Knowledge MCP Server — managed AWS docs, regional availability & region inventory tools",
    targetConfiguration={
        "mcp": {
            "mcpServer": {
                "endpoint": MCP_ENDPOINT
            }
        }
    },
    # Outbound Auth via the gateway service role (SigV4 / IAM).
    credentialProviderConfigurations=[
        {
            "credentialProviderType": "GATEWAY_IAM_ROLE",
            "credentialProvider": {
                "iamCredentialProvider": {
                    "service": "bedrock-agentcore",
                    "region": REGION,      # optional — defaults to gateway's region
                }
            },
        }
    ],
)
target_id = create_target_response["targetId"]
print("Target ID  :", target_id)

# Use this ARN as the connector_id in Step 1
connector_id = gateway_arn
print("\nUse as connector_id ->", connector_id)

### Step 0b: Wait for the Target to become READY

The gateway was already confirmed `READY` before the target was created (a target cannot be added
while the gateway is still `CREATING`). Because the target uses **MCP listing mode = Default**, its
tools are cached at the control plane and it must finish a **sync** before the agent can discover them.
Poll `get_gateway_target` until it reports `READY`. `CREATING`/`UPDATING`/`SYNCHRONIZING` are transient;
`FAILED` is a terminal error.

In [ ]:
import time

# Gateway readiness was already confirmed before the target was created.
# The target uses MCP listing mode = Default, so wait for it to finish syncing.
READY = "READY"
FAILED_STATES = {"FAILED", "DELETING", "DELETE_FAILED"}

def _wait_target_ready(timeout=300, interval=10):
    deadline = time.time() + timeout
    while True:
        resp = gateway_client.get_gateway_target(gatewayIdentifier=gateway_id, targetId=target_id)
        status = resp["status"]
        print(f"Target status: {status}")
        if status == READY:
            return resp
        if status in FAILED_STATES:
            raise RuntimeError(f"Target failed: {status} -> {resp.get('statusReasons', resp)}")
        if time.time() > deadline:
            raise TimeoutError(f"Target not READY after {timeout}s (last status: {status})")
        time.sleep(interval)

_wait_target_ready()
print("\nGateway and target are READY — safe to run the agent.")

## Step 1: Create Agent with MCP Tools

Define an agent with multiple MCP connectors:

**HostedMCPTool Configuration**:
- `type`: Always `"mcp"` for MCP connectors
- `server_label`: Human-readable name for the MCP server
- `connector_id`: AWS Bedrock MCP gateway ARN (format: `arn:aws:bedrock-agentcore:region:account:gateway/name`)
- `server_description`: Brief description of what this MCP provides
- `allowed_tools`: (Optional) List of specific tools to expose from this MCP
- `require_approval`: `"never"` (auto-execute) or `"always"` (human-in-loop)

💡 **Multi-MCP**: Agents can use multiple MCP connectors simultaneously for different services.

In [ ]:
agent = Agent(
    name="Assistance",
    instructions="Provide answers to the queries using tools",
    model=model_id,
    tools=[
        HostedMCPTool(tool_config=        
        {
            "type": "mcp",
            "server_label": "aws-mcp",
            "connector_id": connector_id,
            "server_description": "aws mcp",
            "require_approval": "never",     
        })
    ]
)

## Step 2: Query AWS MCP

Ask a question that requires the AWS MCP tool:

⚡ **Behind the Scenes**: Agent discovers AWS Regions tool from aws-mcp connector and uses it to fetch current data.

🎯 **Result**: Live data from AWS services via MCP!

In [ ]:
result = await Runner.run(agent,"What AWS Regions are available today?")
print(result.final_output)

## Step 3: Stream Response with MCP Tool Execution

Use `Runner.run_streamed()` and process events as they arrive.

In [ ]:
result = Runner.run_streamed(agent,"What AWS Regions are available today?")
async for event in result.stream_events():
    # Token-level text deltas come through as raw_response_event
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)
print()

## 🎉 Congratulations!

You've completed the **MCP Integration** notebook!